In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from glob import glob
import time
import torchvision # Added torchvision import for version checking

print(f"PyTorch Version: {torch.__version__}")
print(f"torchvision Version: {torchvision.__version__}") # Corrected to use torchvision.__version__

PyTorch Version: 2.9.0+cu126
torchvision Version: 0.24.0+cu126


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# IMPORTANT: Update these variables!
NUM_CLASSES = 8
MODEL_WEIGHTS_PATH = "/content/drive/MyDrive/medai-hardik/model-checkpoints/best_swin.pth" # e.g., if you uploaded it to the root
DATASET_PATH = "/content/drive/MyDrive/medai-hardik/balanced_augmented_dataset" # e.g., if you uploaded a folder named 'test_images'

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Configuration loaded. Model will run on: {DEVICE}")
print(f"Expecting weights at: {MODEL_WEIGHTS_PATH}")
if not os.path.exists(MODEL_WEIGHTS_PATH):
    print(f"WARNING: Model weights not found at {MODEL_WEIGHTS_PATH}. Please ensure the file is uploaded or the path is correct.")
print(f"Expecting images in: {DATASET_PATH}")
if not os.path.exists(DATASET_PATH):
    print(f"WARNING: Dataset path not found at {DATASET_PATH}. Please ensure the folder exists or the path is correct.")

Configuration loaded. Model will run on: cuda:0
Expecting weights at: /content/drive/MyDrive/medai-hardik/model-checkpoints/best_swin.pth
Expecting images in: /content/drive/MyDrive/medai-hardik/balanced_augmented_dataset


In [ ]:
# Cell 3: CBAM (Convolutional Block Attention Module) Implementation
# --------------------------------------------------------------------------

class ChannelAttention(nn.Module):
    """Channel Attention Module (CAM) for CBAM."""
    def __init__(self, in_channels, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    """Spatial Attention Module (SAM) for CBAM."""
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1

        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Apply average and max pooling across the channel dimension
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        # Concatenate the pooled outputs
        x_concat = torch.cat([avg_out, max_out], dim=1)

        # Apply convolution and sigmoid
        out = self.conv(x_concat)
        return self.sigmoid(out)

class CBAM(nn.Module):
    """Full CBAM module: CAM followed by SAM."""
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_channels, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        # 1. Channel Attention
        x_ca = x * self.ca(x)
        # 2. Spatial Attention
        x_sa = x_ca * self.sa(x_ca)
        return x_sa


# %% [code]
# --------------------------------------------------------------------------
# Cell 4: HyperColumn-CBAM-DenseNet169 Model Architecture
# --------------------------------------------------------------------------

class HyperColumnCBAMDenseNet169(nn.Module):
    """
    Combines DenseNet169 (Backbone) with HyperColumns (Multi-scale Context) and CBAM (Attention).
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super(HyperColumnCBAMDenseNet169, self).__init__()

        # Load pre-trained DenseNet169 backbone
        densenet = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)

        # Separate the feature extractor (Dense Blocks) from the classifier (final linear layer)
        self.features = densenet.features

        # --- Define Sequential Blocks for Chaining in Forward Pass ---
        self.init_conv = nn.Sequential(self.features.conv0, self.features.norm0, self.features.relu0, self.features.pool0)
        self.db1 = self.features.denseblock1
        self.t1 = self.features.transition1  # HC Source 1 (Output channels: 128)
        self.db2 = self.features.denseblock2
        self.t2 = self.features.transition2  # HC Source 2 (Output channels: 256)
        self.db3 = self.features.denseblock3
        self.t3 = self.features.transition3  # HC Source 3 (Output channels: 640)
        self.db4 = self.features.denseblock4
        self.norm_final = self.features.norm5 # Final normalization layer

        # Calculate the total number of channels after concatenating the HyperColumns
        HC_FUSION_CHANNELS = (self.norm_final.num_features +  # 1664
                              self.t3.conv.out_channels +      # 640
                              self.t2.conv.out_channels +      # 256
                              self.t1.conv.out_channels)       # 128
        # Total: 2688

        # Fusion layer to process the combined HyperColumn features
        self.fusion_conv = nn.Conv2d(HC_FUSION_CHANNELS, 1024, kernel_size=1, bias=False)
        self.bn_fusion = nn.BatchNorm2d(1024)

        # Apply CBAM to the fused features
        self.cbam = CBAM(1024)

        # Global pooling and final classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes) # Map fused channels to the number of classes
        )

    def forward(self, x):

        # 1. Initial layers
        x = self.init_conv(x)

        # 2. Dense Block 1 -> Transition 1 (HC Source 1)
        x = self.db1(x)
        t1_out = self.t1(x)

        # 3. Dense Block 2 -> Transition 2 (HC Source 2)
        x = self.db2(t1_out)
        t2_out = self.t2(x)

        # 4. Dense Block 3 -> Transition 3 (HC Source 3)
        x = self.db3(t2_out)
        t3_out = self.t3(x)

        # 5. Final Dense Block 4 -> Final Norm (HC Source 4)
        x = self.db4(t3_out)
        x_final = self.norm_final(x)

        # --- HyperColumn FUSION ---
        # Upsample all intermediate features to the size of the final feature map (x_final)
        upsample_target_size = x_final.shape[2:]

        # We use the outputs of the Transition layers for the HC features
        t1_resized = F.interpolate(t1_out, size=upsample_target_size, mode='bilinear', align_corners=False)
        t2_resized = F.interpolate(t2_out, size=upsample_target_size, mode='bilinear', align_corners=False)
        t3_resized = F.interpolate(t3_out, size=upsample_target_size, mode='bilinear', align_corners=False)

        # 2. Concatenate the features along the channel dimension
        hypercolumn_features = torch.cat([x_final, t3_resized, t2_resized, t1_resized], dim=1)

        # 3. Fusion Convolution to reduce channel count
        fused_features = F.relu(self.bn_fusion(self.fusion_conv(hypercolumn_features)))

        # 4. Apply CBAM Attention
        attention_output = self.cbam(fused_features)

        # 5. Global Pooling
        out = self.avgpool(attention_output)

        # 6. Final Classification
        out = torch.flatten(out, 1)
        out = self.classifier(out)

        return out


# %% [code]
# --------------------------------------------------------------------------
# Cell 5: Dataset, Dataloader, and Inference Function
# --------------------------------------------------------------------------

class FractureImageDataset(Dataset):
    """Simple Dataset to load images from a folder."""
    def __init__(self, root_dir, transform=None):
        # We look for common image extensions recursively
        self.image_paths = sorted(glob(os.path.join(root_dir, '**', '*.jpg'), recursive=True) +
                                  glob(os.path.join(root_dir, '**', '*.jpeg'), recursive=True) +
                                  glob(os.path.join(root_dir, '**', '*.png'), recursive=True))

        if not self.image_paths and not os.path.exists(root_dir):
            print(f"ERROR: Root directory '{root_dir}' does not exist. Using simulated data.")
            self.image_paths = [f"simulated_image_{i}.jpg" for i in range(5)]
            self.simulated = True
        elif not self.image_paths:
            print(f"WARNING: No images found in '{root_dir}'. Using simulated data.")
            self.image_paths = [f"simulated_image_{i}.jpg" for i in range(5)]
            self.simulated = True
        else:
            self.simulated = False

        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        if self.simulated:
             # Create a dummy tensor if running in a dry-run
            image = Image.fromarray(np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8))
        else:
            try:
                image = Image.open(path).convert("RGB")
            except Exception as e:
                print(f"Error loading image {path}: {e}. Replacing with dummy image.")
                image = Image.fromarray(np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8))

        if self.transform:
            image_tensor = self.transform(image)
        else:
            image_tensor = transforms.ToTensor()(image)

        return image_tensor, path

def run_inference(model, dataloader, device):
    """Runs inference on the provided data loader."""
    model.eval()
    all_predictions = []
    all_paths = []

    print(f"Starting inference on {len(dataloader.dataset)} images...")
    start_time = time.time()

    with torch.no_grad():
        for i, (inputs, paths) in enumerate(dataloader):
            inputs = inputs.to(device)
            outputs = model(inputs)

            # Get probabilities and predicted class index
            # We skip softmax here if we only need the predicted class
            _, predicted_class = torch.max(outputs, 1)

            # Store results
            all_predictions.extend(predicted_class.cpu().numpy())
            all_paths.extend(paths)

            # Simple progress update
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(dataloader)} batches.")

    end_time = time.time()
    print(f"Inference took {end_time - start_time:.2f} seconds.")
    return all_paths, all_predictions


# %% [code]
# --------------------------------------------------------------------------
# Cell 6: Main Execution Block (Model Loading and Inference)
# --------------------------------------------------------------------------

if __name__ == '__main__':
    print(f"--- HyperColumn-CBAM-DenseNet169 Inference ---")
    print(f"Using device: {DEVICE}")

    # 1. Define Standard Transformations
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 2. Setup Data Loading
    test_dataset = FractureImageDataset(DATASET_PATH, transform=test_transform)
    test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

    # 3. Instantiate the Model
    model = HyperColumnCBAMDenseNet169(num_classes=NUM_CLASSES)
    model.to(DEVICE)

    # 4. Load Pre-trained Weights
    try:
        print(f"Attempting to load weights from: {MODEL_WEIGHTS_PATH}")
        model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
        print("Model weights loaded successfully.")
    except FileNotFoundError:
        print(f"WARNING: Weights file not found at {MODEL_WEIGHTS_PATH}. Using untrained model (random weights).")
    except Exception as e:
        print(f"ERROR loading weights: {e}. Check if the saved state_dict matches the current model architecture.")

    # 5. Run Inference
    if len(test_dataset.image_paths) > 0:
        paths, predictions = run_inference(model, test_dataloader, DEVICE)

        # 6. Display Results
        class_labels = {
            0: 'Healthy', 1: 'Comminuted', 2: 'Oblique Displaced',
            3: 'Transverse', 4: 'Spiral', 5: 'Greenstick', 6: 'Impacted',
            7: 'Pathologic'
        }

        print("\n--- Sample Prediction Results ---")
        for i in range(min(10, len(paths))):
            predicted_label = class_labels.get(predictions[i], 'Unknown')
            path_display = os.path.basename(paths[i]) if not test_dataset.simulated else paths[i]
            print(f"Image: {path_display:<30} | Predicted Class ID: {predictions[i]} | Label: {predicted_label}")

        print(f"\nInference complete for {len(paths)} images.")

    else:
        print("Cannot run inference: Dataset is empty or paths are misconfigured.")

--- HyperColumn-CBAM-DenseNet169 Inference ---
Using device: cuda:0
Downloading: "https://download.pytorch.org/models/densenet169-b2777c0a.pth" to /root/.cache/torch/hub/checkpoints/densenet169-b2777c0a.pth


100%|██████████| 54.7M/54.7M [00:00<00:00, 197MB/s]


Attempting to load weights from: /content/drive/MyDrive/medai-hardik/model-checkpoints/best_swin.pth
ERROR loading weights: Error(s) in loading state_dict for HyperColumnCBAMDenseNet169:
	Missing key(s) in state_dict: "features.conv0.weight", "features.norm0.weight", "features.norm0.bias", "features.norm0.running_mean", "features.norm0.running_var", "features.denseblock1.denselayer1.norm1.weight", "features.denseblock1.denselayer1.norm1.bias", "features.denseblock1.denselayer1.norm1.running_mean", "features.denseblock1.denselayer1.norm1.running_var", "features.denseblock1.denselayer1.conv1.weight", "features.denseblock1.denselayer1.norm2.weight", "features.denseblock1.denselayer1.norm2.bias", "features.denseblock1.denselayer1.norm2.running_mean", "features.denseblock1.denselayer1.norm2.running_var", "features.denseblock1.denselayer1.conv2.weight", "features.denseblock1.denselayer2.norm1.weight", "features.denseblock1.denselayer2.norm1.bias", "features.denseblock1.denselayer2.norm1.runn

In [ ]:
# --------------------------------------------------------------------------
# --- Single Cell: HyperColumn-CBAM-DenseNet169 Training Pipeline ---
# --------------------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torchvision import models, transforms, datasets
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import time
import copy
from glob import glob

# --- 1. Configuration and Hyperparameters ---
NUM_CLASSES = 8
# --- IMPORTANT: UPDATE THESE PATHS ---
TRAIN_DIR = "/content/drive/MyDrive/medai-hardik/balanced_augmented_dataset/train"  # Path to training data root
VAL_DIR = "/content/drive/MyDrive/medai-hardik/balanced_augmented_dataset/val"      # Path to validation data root
MODEL_SAVE_PATH = "./best_hypercolumn_cbam_densenet169.pth"
# --- Training Hyperparameters ---
LEARNING_RATE = 1e-4
BATCH_SIZE = 16
NUM_EPOCHS = 20
WEIGHT_DECAY = 1e-5

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Configuration loaded. Device: {DEVICE}")

# --- 2. CBAM (Convolutional Block Attention Module) Implementation ---

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_concat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_concat)
        return self.sigmoid(out)

class CBAM(nn.Module):
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_channels, ratio)
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        x_ca = x * self.ca(x)
        x_sa = x_ca * self.sa(x_ca)
        return x_sa

# --- 3. HyperColumn-CBAM-DenseNet169 Model Architecture ---

class HyperColumnCBAMDenseNet169(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(HyperColumnCBAMDenseNet169, self).__init__()
        densenet = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)
        self.features = densenet.features

        # Define Sequential Blocks for Chaining
        self.init_conv = nn.Sequential(self.features.conv0, self.features.norm0, self.features.relu0, self.features.pool0)
        self.db1 = self.features.denseblock1
        self.t1 = self.features.transition1  # HC Source 1 (128 channels)
        self.db2 = self.features.denseblock2
        self.t2 = self.features.transition2  # HC Source 2 (256 channels)
        self.db3 = self.features.denseblock3
        self.t3 = self.features.transition3  # HC Source 3 (640 channels)
        self.db4 = self.features.denseblock4
        self.norm_final = self.features.norm5 # Final normalization layer (1664 channels)

        # Calculate total channels for fusion: 1664 + 640 + 256 + 128 = 2688
        HC_FUSION_CHANNELS = (self.norm_final.num_features + self.t3.conv.out_channels +
                              self.t2.conv.out_channels + self.t1.conv.out_channels)

        self.fusion_conv = nn.Conv2d(HC_FUSION_CHANNELS, 1024, kernel_size=1, bias=False)
        self.bn_fusion = nn.BatchNorm2d(1024)
        self.cbam = CBAM(1024)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(1024, num_classes))

    def forward(self, x):
        x = self.init_conv(x)
        x = self.db1(x); t1_out = self.t1(x) # HC Source 1
        x = self.db2(t1_out); t2_out = self.t2(x) # HC Source 2
        x = self.db3(t2_out); t3_out = self.t3(x) # HC Source 3
        x = self.db4(t3_out); x_final = self.norm_final(x) # HC Source 4

        # HyperColumn Fusion
        upsample_target_size = x_final.shape[2:]
        t1_resized = F.interpolate(t1_out, size=upsample_target_size, mode='bilinear', align_corners=False)
        t2_resized = F.interpolate(t2_out, size=upsample_target_size, mode='bilinear', align_corners=False)
        t3_resized = F.interpolate(t3_out, size=upsample_target_size, mode='bilinear', align_corners=False)

        hypercolumn_features = torch.cat([x_final, t3_resized, t2_resized, t1_resized], dim=1)

        fused_features = F.relu(self.bn_fusion(self.fusion_conv(hypercolumn_features)))
        attention_output = self.cbam(fused_features)

        out = self.avgpool(attention_output)
        out = torch.flatten(out, 1)
        out = self.classifier(out)
        return out

# --- 4. Training Function Definition ---

def train_model(model, dataloaders, dataset_sizes, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}\n' + '-' * 10)

        for phase in ['train', 'val']:
            if dataloaders[phase] is None or dataset_sizes[phase] == 0:
                print(f"Skipping {phase} phase: Data not loaded or empty.")
                continue

            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), MODEL_SAVE_PATH)
                print(f"Model saved to {MODEL_SAVE_PATH} (Acc: {best_acc:.4f})")

    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model

# --- 5. Data Preparation and Execution ---

if __name__ == '__main__':
    # Define Standard Normalization
    NORM_MEAN = [0.485, 0.456, 0.406]
    NORM_STD = [0.229, 0.224, 0.225]

    # Transformations
    train_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomRotation(15),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD)
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD)
    ])

    # Load data using ImageFolder
    try:
        image_datasets = {
            'train': datasets.ImageFolder(TRAIN_DIR, train_transforms),
            'val': datasets.ImageFolder(VAL_DIR, val_transforms)
        }
        dataloaders = {
            'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
            'val': DataLoader(image_datasets['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        }
        dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
        print(f"Data Loaded: Train images={dataset_sizes['train']}, Val images={dataset_sizes['val']}")

    except Exception as e:
        print(f"ERROR: Could not load data. Check TRAIN_DIR ({TRAIN_DIR}) and VAL_DIR ({VAL_DIR}).")
        print(f"PyTorch Error: {e}")
        dataloaders = {'train': None, 'val': None}
        dataset_sizes = {'train': 0, 'val': 0}

    # Start training only if data is available
    if dataset_sizes['train'] > 0:
        model_ft = HyperColumnCBAMDenseNet169(num_classes=NUM_CLASSES)
        model_ft = model_ft.to(DEVICE)

        criterion = nn.CrossEntropyLoss()
        optimizer_ft = optim.AdamW(model_ft.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

        print("\nStarting Training...")
        final_model = train_model(model_ft, dataloaders, dataset_sizes, criterion, optimizer_ft, exp_lr_scheduler)
        print("\nTraining Finished. The best model weights are saved.")
    else:
        print("Training cannot start. Please fix data paths.")

Configuration loaded. Device: cuda:0
Data Loaded: Train images=1952, Val images=106

Starting Training...

Epoch 1/20
----------
train Loss: 1.4287 Acc: 0.5471
val Loss: 0.9211 Acc: 0.6887
Model saved to ./best_hypercolumn_cbam_densenet169.pth (Acc: 0.6887)

Epoch 2/20
----------
train Loss: 0.4292 Acc: 0.8796
val Loss: 0.6365 Acc: 0.8113
Model saved to ./best_hypercolumn_cbam_densenet169.pth (Acc: 0.8113)

Epoch 3/20
----------
train Loss: 0.2293 Acc: 0.9355
val Loss: 0.4456 Acc: 0.8774
Model saved to ./best_hypercolumn_cbam_densenet169.pth (Acc: 0.8774)

Epoch 4/20
----------
train Loss: 0.1168 Acc: 0.9713
val Loss: 0.5035 Acc: 0.8868
Model saved to ./best_hypercolumn_cbam_densenet169.pth (Acc: 0.8868)

Epoch 5/20
----------
train Loss: 0.0706 Acc: 0.9846
val Loss: 0.3659 Acc: 0.8962
Model saved to ./best_hypercolumn_cbam_densenet169.pth (Acc: 0.8962)

Epoch 6/20
----------
train Loss: 0.1053 Acc: 0.9688
val Loss: 0.4079 Acc: 0.8962

Epoch 7/20
----------
train Loss: 0.0585 Acc: 0.98

In [5]:
# --------------------------------------------------------------------------
# HyperColumn-CBAM-DenseNet169 Training Pipeline (Colab Version)
# Modified with Focal Loss and Label Verification
# --------------------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torchvision import models, transforms, datasets
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import time
import copy
from glob import glob

# --- 1. Configuration and Hyperparameters ---
NUM_CLASSES = 8
# --- IMPORTANT: UPDATE THESE PATHS FOR COLAB ---
# Ensure your Drive is mounted: from google.colab import drive; drive.mount('/content/drive')
TRAIN_DIR = "/content/drive/MyDrive/medai-hardik/balanced_augmented_dataset/train"  # Path to training data root
VAL_DIR = "/content/drive/MyDrive/medai-hardik/balanced_augmented_dataset/val"      # Path to validation data root
MODEL_SAVE_PATH = "./best_hypercolumn_cbam_densenet169_focal.pth"

# --- Training Hyperparameters ---
LEARNING_RATE = 1e-4
BATCH_SIZE = 16
NUM_EPOCHS = 20
WEIGHT_DECAY = 1e-5

# Focal Loss Hyperparameters
FOCAL_GAMMA = 2.0  # Focuses more on hard examples (misclassified)
FOCAL_ALPHA = 1.0  # Balance factor

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Configuration loaded. Device: {DEVICE}")

# --- 2. Loss Function: Focal Loss ---
# Added to address class imbalance and hard-to-classify examples (e.g., Displaced vs Non-Displaced)

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Standard Cross Entropy Loss
        CE_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-CE_loss) # pt is the probability of the true class

        # Focal Loss Formula: -alpha * (1-pt)^gamma * log(pt)
        F_loss = self.alpha * (1-pt)**self.gamma * CE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

# --- 3. CBAM (Convolutional Block Attention Module) Implementation ---

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_concat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_concat)
        return self.sigmoid(out)

class CBAM(nn.Module):
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_channels, ratio)
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        x_ca = x * self.ca(x)
        x_sa = x_ca * self.sa(x_ca)
        return x_sa

# --- 4. HyperColumn-CBAM-DenseNet169 Model Architecture ---

class HyperColumnCBAMDenseNet169(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(HyperColumnCBAMDenseNet169, self).__init__()
        densenet = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)
        self.features = densenet.features

        # Define Sequential Blocks for Chaining
        self.init_conv = nn.Sequential(self.features.conv0, self.features.norm0, self.features.relu0, self.features.pool0)
        self.db1 = self.features.denseblock1
        self.t1 = self.features.transition1  # HC Source 1 (128 channels)
        self.db2 = self.features.denseblock2
        self.t2 = self.features.transition2  # HC Source 2 (256 channels)
        self.db3 = self.features.denseblock3
        self.t3 = self.features.transition3  # HC Source 3 (640 channels)
        self.db4 = self.features.denseblock4
        self.norm_final = self.features.norm5 # Final normalization layer (1664 channels)

        # Calculate total channels for fusion: 1664 + 640 + 256 + 128 = 2688
        HC_FUSION_CHANNELS = (self.norm_final.num_features + self.t3.conv.out_channels +
                              self.t2.conv.out_channels + self.t1.conv.out_channels)

        self.fusion_conv = nn.Conv2d(HC_FUSION_CHANNELS, 1024, kernel_size=1, bias=False)
        self.bn_fusion = nn.BatchNorm2d(1024)
        self.cbam = CBAM(1024)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(1024, num_classes))

    def forward(self, x):
        x = self.init_conv(x)
        x = self.db1(x); t1_out = self.t1(x) # HC Source 1
        x = self.db2(t1_out); t2_out = self.t2(x) # HC Source 2
        x = self.db3(t2_out); t3_out = self.t3(x) # HC Source 3
        x = self.db4(t3_out); x_final = self.norm_final(x) # HC Source 4

        # HyperColumn Fusion
        upsample_target_size = x_final.shape[2:]
        t1_resized = F.interpolate(t1_out, size=upsample_target_size, mode='bilinear', align_corners=False)
        t2_resized = F.interpolate(t2_out, size=upsample_target_size, mode='bilinear', align_corners=False)
        t3_resized = F.interpolate(t3_out, size=upsample_target_size, mode='bilinear', align_corners=False)

        hypercolumn_features = torch.cat([x_final, t3_resized, t2_resized, t1_resized], dim=1)

        fused_features = F.relu(self.bn_fusion(self.fusion_conv(hypercolumn_features)))
        attention_output = self.cbam(fused_features)

        out = self.avgpool(attention_output)
        out = torch.flatten(out, 1)
        out = self.classifier(out)
        return out

# --- 5. Training Function Definition ---

def train_model(model, dataloaders, dataset_sizes, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}\n' + '-' * 10)

        for phase in ['train', 'val']:
            if dataloaders[phase] is None or dataset_sizes[phase] == 0:
                print(f"Skipping {phase} phase: Data not loaded or empty.")
                continue

            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), MODEL_SAVE_PATH)
                print(f"Model saved to {MODEL_SAVE_PATH} (Acc: {best_acc:.4f})")

    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model

# --- 6. Data Preparation and Execution ---

if __name__ == '__main__':
    # Define Standard Normalization
    NORM_MEAN = [0.485, 0.456, 0.406]
    NORM_STD = [0.229, 0.224, 0.225]

    # Transformations
    train_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomRotation(15),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD)
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD)
    ])

    # Load data using ImageFolder
    try:
        image_datasets = {
            'train': datasets.ImageFolder(TRAIN_DIR, train_transforms),
            'val': datasets.ImageFolder(VAL_DIR, val_transforms)
        }

        # --- VERIFY CLASS MAPPING ---
        print("\n--- Class Mapping Verification ---")
        print(f"Class to Index: {image_datasets['train'].class_to_idx}")
        print("Ensure these indices match your local evaluation scripts (usually alphabetical).")
        print("----------------------------------\n")

        dataloaders = {
            'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
            'val': DataLoader(image_datasets['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        }
        dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
        print(f"Data Loaded: Train images={dataset_sizes['train']}, Val images={dataset_sizes['val']}")

    except Exception as e:
        print(f"ERROR: Could not load data. Check TRAIN_DIR ({TRAIN_DIR}) and VAL_DIR ({VAL_DIR}).")
        print(f"PyTorch Error: {e}")
        dataloaders = {'train': None, 'val': None}
        dataset_sizes = {'train': 0, 'val': 0}

    # Start training only if data is available
    if dataset_sizes['train'] > 0:
        model_ft = HyperColumnCBAMDenseNet169(num_classes=NUM_CLASSES)
        model_ft = model_ft.to(DEVICE)

        # --- MODIFICATION: Use Focal Loss instead of CrossEntropyLoss ---
        print(f"Using Focal Loss (gamma={FOCAL_GAMMA}, alpha={FOCAL_ALPHA})")
        criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)

        optimizer_ft = optim.AdamW(model_ft.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

        print("\nStarting Training...")
        final_model = train_model(model_ft, dataloaders, dataset_sizes, criterion, optimizer_ft, exp_lr_scheduler)
        print("\nTraining Finished. The best model weights are saved.")
    else:
        print("Training cannot start. Please fix data paths.")

Configuration loaded. Device: cuda:0

--- Class Mapping Verification ---
Class to Index: {'Comminuted': 0, 'Greenstick': 1, 'Healthy': 2, 'Oblique': 3, 'Oblique_Displaced': 4, 'Spiral': 5, 'Transverse': 6, 'Transverse_Displaced': 7}
Ensure these indices match your local evaluation scripts (usually alphabetical).
----------------------------------

Data Loaded: Train images=1952, Val images=106
Downloading: "https://download.pytorch.org/models/densenet169-b2777c0a.pth" to /root/.cache/torch/hub/checkpoints/densenet169-b2777c0a.pth


100%|██████████| 54.7M/54.7M [00:00<00:00, 105MB/s]


Using Focal Loss (gamma=2.0, alpha=1.0)

Starting Training...

Epoch 1/20
----------
train Loss: 1.1369 Acc: 0.5123
val Loss: 0.6156 Acc: 0.7264
Model saved to ./best_hypercolumn_cbam_densenet169_focal.pth (Acc: 0.7264)

Epoch 2/20
----------
train Loss: 0.2560 Acc: 0.8678
val Loss: 0.3672 Acc: 0.7642
Model saved to ./best_hypercolumn_cbam_densenet169_focal.pth (Acc: 0.7642)

Epoch 3/20
----------
train Loss: 0.1148 Acc: 0.9257
val Loss: 0.2554 Acc: 0.8491
Model saved to ./best_hypercolumn_cbam_densenet169_focal.pth (Acc: 0.8491)

Epoch 4/20
----------
train Loss: 0.0522 Acc: 0.9657
val Loss: 0.2087 Acc: 0.8774
Model saved to ./best_hypercolumn_cbam_densenet169_focal.pth (Acc: 0.8774)

Epoch 5/20
----------
train Loss: 0.0548 Acc: 0.9600
val Loss: 0.2745 Acc: 0.8679

Epoch 6/20
----------
train Loss: 0.0413 Acc: 0.9693
val Loss: 0.2681 Acc: 0.8868
Model saved to ./best_hypercolumn_cbam_densenet169_focal.pth (Acc: 0.8868)

Epoch 7/20
----------
train Loss: 0.0286 Acc: 0.9780
val Loss: 0